##  셀 1: 패키지 설치

In [ ]:
# 기본 패키지
!pip install -q insightface onnxruntime-gpu opencv-python-headless Pillow

# 3DDFA_V2 클론 (facemap_3dmm과 동일한 BFM 기반 3DMM)
!git clone -q https://github.com/cleardusk/3DDFA_V2.git /content/3DDFA_V2
%cd /content/3DDFA_V2
!pip install -q -r requirements.txt

# Cython 가속 모듈 컴파일
!python setup.py build_ext --inplace 2>/dev/null || echo '[INFO] Cython 빌드 실패 -> Python fallback 사용'

print('\n✅ 설치 완료')

##  셀 2: 모델 다운로드

In [ ]:
import os

os.makedirs('/root/.insightface/models', exist_ok=True)

inswapper_path = '/root/.insightface/models/inswapper_128.onnx'
if not os.path.exists(inswapper_path):
    print('⬇ inswapper_128.onnx 다운로드 중... (약 500MB)')
    !wget -q --show-progress \
        -O {inswapper_path} \
        https://huggingface.co/ezioruan/inswapper_128.onnx/resolve/main/inswapper_128.onnx
else:
    print('✅ inswapper_128.onnx 이미 존재')

print('\n✅ 모델 다운로드 완료')

##
 셀 3: 모듈 import 및 InsightFace 초기화

In [ ]:
import sys
import cv2
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

sys.path.insert(0, '/content/3DDFA_V2')

import insightface
from insightface.app import FaceAnalysis

print(f'InsightFace 버전: {insightface.__version__}')

# FaceAnalysis 초기화 (GPU)
face_app = FaceAnalysis(
    name='buffalo_l',
    providers=['CUDAExecutionProvider', 'CPUExecutionProvider']
)
face_app.prepare(ctx_id=0, det_size=(640, 640))

# inswapper 로드
swapper = insightface.model_zoo.get_model(
    '/root/.insightface/models/inswapper_128.onnx',
    download=False
)

print('\n✅ InsightFace 초기화 완료')

## 셀 4: 3DDFA_V2 초기화 (facemap_3dmm 대체)

In [ ]:
import yaml
from FaceBoxes import FaceBoxes
from TDDFA import TDDFA

cfg = yaml.load(
    open('/content/3DDFA_V2/configs/mb1_120x120.yml'),
    Loader=yaml.SafeLoader
)

tddfa = TDDFA(gpu_mode=True, **cfg)
face_boxes_detector = FaceBoxes()

print('✅ 3DDFA_V2 초기화 완료')
print('   출력: 68개 3D 랜드마크 (x,y,z) + Shape/Expression/Pose 계수')

## 셀 5: 핵심 변환 함수 — 3DMM 랜드마크 → InsightFace keypoints

In [ ]:
# ──────────────────────────────────────────────────────────────────────────
# InsightFace 5-point keypoints 인덱스:
#   [0] 오른쪽 눈  [1] 왼쪽 눈  [2] 코끝  [3] 오른쪽 입꼬리  [4] 왼쪽 입꼬리
#
# 68 랜드마크 (iBUG/300W 기준):
#   오른쪽 눈: 36~41, 왼쪽 눈: 42~47, 코끝: 30, 입: 48(우) / 54(좌)
# ──────────────────────────────────────────────────────────────────────────

def landmarks_68_to_5kps(landmarks_3d: np.ndarray) -> np.ndarray:
    """
    68개 3D 랜드마크 -> InsightFace 5-point keypoints (5x2)

    Args:
        landmarks_3d: shape (3, 68) 또는 (68, 3) 또는 (2, 68)
    Returns:
        kps: np.ndarray (5, 2) float32
    """
    lm = np.array(landmarks_3d)
    if lm.shape == (3, 68):
        lm = lm.T
    elif lm.shape == (2, 68):
        lm = np.vstack([lm, np.zeros((1, 68))]).T
    xy = lm[:, :2]

    return np.array([
        xy[36:42].mean(axis=0),   # 오른쪽 눈 (36~41 평균)
        xy[42:48].mean(axis=0),   # 왼쪽 눈  (42~47 평균)
        xy[30],                    # 코끝
        xy[48],                    # 오른쪽 입꼬리
        xy[54],                    # 왼쪽 입꼬리
    ], dtype=np.float32)


def extract_3dmm_features(img_bgr: np.ndarray) -> dict:
    """
    이미지 -> 3DMM 특징 전체 추출
    Returns: landmarks_68_3d, landmarks_5kps, bbox, pose, params, detected
    """
    result = dict(landmarks_68_3d=None, landmarks_5kps=None,
                  bbox=None, pose=None, params=None, detected=False)

    boxes = face_boxes_detector(img_bgr)
    if len(boxes) == 0:
        print('⚠ 얼굴 감지 실패')
        return result

    param_lst, roi_box_lst = tddfa(img_bgr, boxes)
    ver_lst = tddfa.recon_vers(param_lst, roi_box_lst, dense_flag=False)
    lm3d = ver_lst[0]  # (3, 68)

    try:
        from utils.pose import calc_pose
        pose = calc_pose(param_lst[0])
    except Exception:
        pose = (0.0, 0.0, 0.0)

    kps_5 = landmarks_68_to_5kps(lm3d)
    result.update(dict(
        landmarks_68_3d=lm3d.T,
        landmarks_5kps=kps_5,
        bbox=roi_box_lst[0],
        pose=pose,
        params=param_lst[0],
        detected=True
    ))
    return result


def inject_3dmm_kps(face_obj, kps_5: np.ndarray):
    """InsightFace Face 객체의 kps를 3DMM 기반 5점으로 교체"""
    face_obj.kps = kps_5.astype(np.float32)
    return face_obj


print('✅ 변환 함수 정의 완료')

## 셀 6: 메인 파이프라인 — faceswap_pipeline()

In [ ]:
def faceswap_pipeline(
    source_img_bgr: np.ndarray,
    target_img_bgr: np.ndarray,
    use_3dmm_kps: bool = True,
    verbose: bool = True
) -> np.ndarray:
    """
    facemap_3dmm -> InsightFace inswapper 전체 파이프라인

    Args:
        source_img_bgr : 소스 얼굴 이미지 (BGR)
        target_img_bgr : 타겟 이미지 (이 얼굴에 소스 얼굴을 합성)
        use_3dmm_kps   : True = 3DMM 기반 5-kps 주입 (측면/큰 각도에 유리)
        verbose        : 디버그 출력 여부
    Returns:
        result: BGR numpy array
    """
    # Step 1: 소스 얼굴 감지
    src_faces = face_app.get(source_img_bgr)
    if not src_faces:
        raise ValueError('소스 이미지에서 얼굴을 감지하지 못했습니다.')
    src_face = src_faces[0]
    if verbose:
        print(f'  [소스] 얼굴 {len(src_faces)}개 감지')

    # Step 2: 타겟 얼굴 감지
    tgt_faces = face_app.get(target_img_bgr)
    if not tgt_faces:
        raise ValueError('타겟 이미지에서 얼굴을 감지하지 못했습니다.')
    if verbose:
        print(f'  [타겟] 얼굴 {len(tgt_faces)}개 감지')

    # Step 3: 3DMM 특징 추출
    tgt_3dmm = {'detected': False}
    if use_3dmm_kps:
        if verbose:
            print('  [3DMM] 3D 랜드마크 추출 중...')
        tgt_3dmm = extract_3dmm_features(target_img_bgr)
        if tgt_3dmm['detected'] and verbose:
            p = tgt_3dmm['pose']
            print(f'  [3DMM] Pose -> yaw:{p[0]:.1f} pitch:{p[1]:.1f} roll:{p[2]:.1f}')

    # Step 4: 얼굴별 swap 수행
    result = target_img_bgr.copy()
    for i, tgt_face in enumerate(tgt_faces):
        if use_3dmm_kps and tgt_3dmm['detected']:
            tgt_face = inject_3dmm_kps(tgt_face, tgt_3dmm['landmarks_5kps'])
        result = swapper.get(result, tgt_face, src_face, paste_back=True)
        if verbose:
            print(f'  [swap] 얼굴 {i+1}/{len(tgt_faces)} 완료')

    return result


print('✅ faceswap_pipeline 정의 완료')

## 🖼 셀 7: 이미지 업로드

In [ ]:
from google.colab import files

def show_images(imgs, titles, figsize=(18, 6)):
    fig, axes = plt.subplots(1, len(imgs), figsize=figsize)
    if len(imgs) == 1:
        axes = [axes]
    for ax, img, title in zip(axes, imgs, titles):
        ax.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
        ax.set_title(title, fontsize=13)
        ax.axis('off')
    plt.tight_layout()
    plt.show()


print('📁 소스 이미지 업로드 (얼굴을 가져올 이미지)')
src_upload = files.upload()
src_path = list(src_upload.keys())[0]

print('\n📁 타겟 이미지 업로드 (얼굴을 교체할 이미지)')
tgt_upload = files.upload()
tgt_path = list(tgt_upload.keys())[0]

src_img = cv2.imread(src_path)
tgt_img = cv2.imread(tgt_path)

assert src_img is not None, f'소스 이미지 로드 실패: {src_path}'
assert tgt_img is not None, f'타겟 이미지 로드 실패: {tgt_path}'

print(f'\n소스: {src_img.shape[1]}x{src_img.shape[0]}')
print(f'타겟: {tgt_img.shape[1]}x{tgt_img.shape[0]}')
show_images([src_img, tgt_img], ['소스 (이 얼굴을)', '타겟 (여기에 합성)'])

## 🚀 셀 8: Face Swap 실행

In [ ]:
import time

print('=' * 50)
print('🎭 Face Swap 파이프라인 시작')
print('=' * 50)
t0 = time.time()

result_img = faceswap_pipeline(
    source_img_bgr=src_img,
    target_img_bgr=tgt_img,
    use_3dmm_kps=True,   # False = InsightFace 기본 kps 사용
    verbose=True
)

print(f'\n⏱ 처리 시간: {time.time()-t0:.2f}초')
print('=' * 50)

cv2.imwrite('/content/result_faceswap.jpg', result_img)
print('💾 저장: /content/result_faceswap.jpg')

show_images(
    [src_img, tgt_img, result_img],
    ['소스 얼굴', '타겟 원본', '✅ Face Swap 결과']
)

## 🔍 셀 9: 3DMM 출력 데이터 확인 (랜드마크 시각화)

In [ ]:
print('3DMM 출력 데이터 분석')
print('=' * 50)

for name, img in [('소스', src_img), ('타겟', tgt_img)]:
    print(f'\n[{name}]')
    feat = extract_3dmm_features(img)
    if not feat['detected']:
        print('  얼굴 없음')
        continue
    lm = feat['landmarks_68_3d']
    print(f'  68개 3D 랜드마크 shape: {lm.shape}')
    print(f'  x: {lm[:,0].min():.1f} ~ {lm[:,0].max():.1f}')
    print(f'  y: {lm[:,1].min():.1f} ~ {lm[:,1].max():.1f}')
    print(f'  z: {lm[:,2].min():.1f} ~ {lm[:,2].max():.1f}')
    print(f'  Pose (yaw/pitch/roll): {feat["pose"]}')
    print(f'  3DMM params shape: {feat["params"].shape}')
    kps_labels = ['우측눈', '좌측눈', '코끝', '우측입', '좌측입']
    print('  5-kps:')
    for label, pt in zip(kps_labels, feat['landmarks_5kps']):
        print(f'    {label}: ({pt[0]:.1f}, {pt[1]:.1f})')

# 타겟 이미지에 랜드마크 오버레이
feat = extract_3dmm_features(tgt_img)
if feat['detected']:
    overlay = tgt_img.copy()
    for x, y, z in feat['landmarks_68_3d']:
        cv2.circle(overlay, (int(x), int(y)), 2, (0, 255, 0), -1)
    for pt in feat['landmarks_5kps']:
        cv2.circle(overlay, (int(pt[0]), int(pt[1])), 7, (0, 0, 255), -1)
    show_images(
        [tgt_img, overlay],
        ['타겟 원본', '68 랜드마크 (녹색) + 5-kps (빨강)']
    )

## ✨ 셀 10: (선택) GFPGAN 얼굴 복원으로 품질 향상

In [ ]:
!pip install -q gfpgan
from gfpgan import GFPGANer

restorer = GFPGANer(
    model_path='https://github.com/TencentARC/GFPGAN/releases/download/v1.3.0/GFPGANv1.4.pth',
    upscale=1,
    arch='clean',
    channel_multiplier=2
)

print('GFPGAN 복원 중...')
_, _, restored = restorer.enhance(
    result_img,
    has_aligned=False,
    only_center_face=False,
    paste_back=True
)

cv2.imwrite('/content/result_faceswap_restored.jpg', restored)
show_images([result_img, restored], ['Face Swap', '✅ Face Swap + GFPGAN'])
print('💾 저장: /content/result_faceswap_restored.jpg')

## 💾 셀 11: 결과 다운로드

In [ ]:
import os
from google.colab import files

files.download('/content/result_faceswap.jpg')
if os.path.exists('/content/result_faceswap_restored.jpg'):
    files.download('/content/result_faceswap_restored.jpg')
print('✅ 다운로드 완료')

## 🎬 셀 12: (확장) 비디오 처리 파이프라인

In [ ]:
import time

def process_video(
    source_img_bgr: np.ndarray,
    video_path: str,
    output_path: str = '/content/result_video.mp4',
    use_3dmm_kps: bool = True,
    max_frames: int = None
):
    """
    비디오 전체 프레임에 face swap 적용 (실시간 파이프라인)
    """
    cap = cv2.VideoCapture(video_path)
    fps    = cap.get(cv2.CAP_PROP_FPS)
    width  = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    total  = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

    out = cv2.VideoWriter(
        output_path,
        cv2.VideoWriter_fourcc(*'mp4v'),
        fps, (width, height)
    )

    print(f'비디오: {total}프레임 @ {fps:.1f}fps ({width}x{height})')

    src_faces = face_app.get(source_img_bgr)
    assert src_faces, '소스에서 얼굴 감지 실패'
    src_face = src_faces[0]

    frame_count = 0
    t0 = time.time()

    while True:
        ret, frame = cap.read()
        if not ret or (max_frames and frame_count >= max_frames):
            break

        tgt_faces = face_app.get(frame)
        result = frame.copy()

        if tgt_faces:
            for tgt_face in tgt_faces:
                if use_3dmm_kps:
                    feat = extract_3dmm_features(frame)
                    if feat['detected']:
                        tgt_face = inject_3dmm_kps(tgt_face, feat['landmarks_5kps'])
                result = swapper.get(result, tgt_face, src_face, paste_back=True)

        out.write(result)
        frame_count += 1

        if frame_count % 30 == 0:
            elapsed = time.time() - t0
            print(f'  {frame_count}/{total} ({elapsed:.1f}s, {frame_count/elapsed:.1f} fps)')

    cap.release()
    out.release()
    print(f'완료: {output_path} ({frame_count}프레임, {time.time()-t0:.1f}초)')


# 아래 주석 해제 후 사용:
# vid_upload = files.upload()
# vid_path = list(vid_upload.keys())[0]
# process_video(src_img, vid_path, max_frames=100)
# files.download('/content/result_video.mp4')

print('✅ 비디오 파이프라인 준비 완료 (주석 해제 후 실행)')